# 分箱约束策略与统一评价目标

当前统一目标入口支持 `or_tools`、`cp_sat`、`genetic`。`objective` 可以接收原生指标名、8 个预定义复合指标名，或评价函数；`objective_params` 提供权重/附加参数。

`constraint_policy="raise"` 为默认严格模式；`"legacy"` 在约束不可行时保留原生尽力满足结果或优化器的单箱回退。输入错误和回调异常仍然报错；超时沿用原有可行解/异常处理，固定用户规则保持优先级。

本例使用仓库现有验收解释器与 `examples/hscredit_yyp.xlsx`。JSON 只保存聚合核验结果，不保存原始客户行。

In [1]:
from pathlib import Path
from importlib import metadata
from xml.etree import ElementTree as ET
import hashlib, json, sys
import numpy as np
import pandas as pd
from sklearn.base import clone
import hscredit
from hscredit.core.binning import OptimalBinning, OptimalBinning2D, CustomObjectives
from hscredit.core.metrics import compute_bin_stats

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'hscredit/core/binning').is_dir())
assert Path(hscredit.__file__).resolve().parents[1] == ROOT
OUT = ROOT/'docs/validation'
OUT.mkdir(exist_ok=True)
FEATURES=['衡枢鉴真分老客版','近六个月非银多头机构数','青云24']
DATA=pd.read_excel(ROOT/'examples/hscredit_yyp.xlsx',usecols=FEATURES+['商品类别','FPD'])
assert DATA.FPD.isin([0,1]).all()
EVIDENCE={'日期':'2026-10-27','时区':'Asia/Shanghai','解释器':sys.executable,
          '版本':{n:metadata.version(n) for n in ['hscredit','numpy','pandas','scikit-learn','ortools','joblib','cloudpickle']},
          '样本数':len(DATA),'好样本数':int(DATA.FPD.eq(0).sum()),'坏样本数':int(DATA.FPD.eq(1).sum()),
          '源码SHA256':{str(p.relative_to(ROOT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted((ROOT/'hscredit/core/binning').glob('*.py'))}}
def clean(value):
    if isinstance(value,dict): return {str(k):clean(v) for k,v in value.items()}
    if isinstance(value,(list,tuple,np.ndarray)): return [clean(v) for v in value]
    if isinstance(value,np.generic): return clean(value.item())
    if isinstance(value,float) and not np.isfinite(value): return None
    return value

def save():
    (OUT/'binning-objectives-and-constraints-2026-10-27.json').write_text(json.dumps(clean(EVIDENCE),ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
print('样本数：',len(DATA),'坏样本数：',EVIDENCE['坏样本数'])
print('复合目标：',CustomObjectives.names())
print('CP-SAT 接受的目标：',OptimalBinning.available_objectives('cp_sat'))


样本数： 970 坏样本数： 136
复合目标： ['max_lift_iv', 'min_lift_iv', 'max_min_lift_sum_iv', 'max_lift_distance_iv', 'min_lift_distance_iv', 'lift_distance_sum_iv', 'max_ks_iv', 'woe_variance_iv']
CP-SAT 接受的目标： ['iv', 'ks', 'gini', 'max_lift_iv', 'min_lift_iv', 'max_min_lift_sum_iv', 'max_lift_distance_iv', 'min_lift_distance_iv', 'lift_distance_sum_iv', 'max_ks_iv', 'woe_variance_iv']


## 1. 按名称选择目标

以下命名目标会最大化 `IV + 0.2 × 最大 Lift`。字符串只解析公开注册名称，不执行表达式。


In [2]:
named=OptimalBinning(method='or_tools',objective='max_lift_iv',
    objective_params={'iv_weight':1.0,'lift_weight':0.2},constraint_policy='raise',
    max_n_bins=3,min_bin_size=.05,n_prebins=8,time_limit=5,n_jobs=1).fit(DATA[[FEATURES[0]]],DATA.FPD)
print(named.get_bin_table(FEATURES[0])[['分箱标签','样本总数','坏样本数','坏样本率','分档IV值','LIFT值']].to_string(index=False))
print('实际目标：',named._binner.solver_info_[FEATURES[0]]['目标函数'])


            分箱标签  样本总数  坏样本数   坏样本率  分档IV值  LIFT值
  [-inf, 0.0697)   364    32 0.0879 0.0856 0.6270
[0.0697, 0.1581)   484    67 0.1384 0.0001 0.9873
  [0.1581, +inf)   122    37 0.3033 0.1671 2.1631
实际目标： max_lift_iv


C:\Users\Administrator\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] 系统找不到指定的文件。
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\Administrator\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\Administrator\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Administrator\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\Administrator\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi

## 2. 同一个参数传入自定义函数

回调接收按箱号排序的统计字典列表、总体好数、总体坏数，并返回有限实数，越大越好。`objective_params` 作为额外关键字参数传入。`bad_rate` 是箱内坏率，评价 IV 时直接使用 `iv` 或 `分档IV值`。

CP-SAT 类接到任意 Python 回调后使用有时限的候选枚举，实际后端记录为 `enumeration`。

In [3]:
def my_objective(bin_stats,total_good,total_bad,penalty=.01):
    assert sum(s['count'] for s in bin_stats)==total_good+total_bad
    return sum(s['分档IV值'] for s in bin_stats)-penalty*len(bin_stats)

custom=OptimalBinning(method='cp_sat',objective=my_objective,objective_params={'penalty':.02},
    max_n_bins=3,min_bin_size=.05,n_prebins=8,time_limit=5,n_jobs=1).fit(DATA[[FEATURES[0]]],DATA.FPD)
assert custom.solver_info_[FEATURES[0]]['后端']=='enumeration'
assert clone(custom).objective is my_objective
print('自定义函数结果：',custom.splits_[FEATURES[0]].tolist())
print('求解信息：',custom.solver_info_[FEATURES[0]])


自定义函数结果： [0.06969581326685517, 0.1580825857217315]
求解信息： {'后端': 'enumeration', '求解状态': 'OPTIMAL', '目标函数': 'my_objective', '目标值': 0.19275739307065648, '耗时秒': 0.03471579996403307}


## 3. 选择约束冲突策略

这个可手算样本的坏率随特征严格增加，要求至少 3 箱且坏率递减不可行。严格模式报错；兼容模式保留原生规则，`constraint_violations_` 记录未满足项。普通分箱表不添加状态或原因列。

In [4]:
toy=pd.DataFrame({'数值':np.repeat(np.arange(6,dtype=float),20)})
toy_y=pd.Series(np.concatenate([np.r_[np.ones(k),np.zeros(20-k)] for k in [2,4,6,10,14,18]]).astype(int))
policies=[]
for policy in ['raise','legacy']:
    row={'约束策略':policy}
    try:
        model=OptimalBinning(method='tree',min_n_bins=3,max_n_bins=3,monotonic='descending',constraint_policy=policy,n_jobs=1).fit(toy,toy_y)
        row.update({'实际箱数':model.n_bins_['数值'],'规则':model.splits_['数值'].tolist(),'未满足项':model.constraint_violations_})
        assert len(model.transform(toy))==len(toy)
    except ValueError as error:
        row['异常']=str(error)
    policies.append(row)
assert '异常' in policies[0] and '异常' not in policies[1]
assert policies[1]['未满足项']
EVIDENCE['约束策略对照']=policies
print(json.dumps(policies,ensure_ascii=False,indent=2))


[
  {
    "约束策略": "raise",
    "异常": "特征 '数值' 的约束 min_n_bins=3, max_n_bins=3, min_bin_size=0.01, max_bin_size=None, monotonic=descending 无法满足: 候选边界上不存在同时满足箱数、样本数、坏率及单调约束的可行分箱"
  },
  {
    "约束策略": "legacy",
    "实际箱数": 3,
    "规则": [
      2.5,
      3.5
    ],
    "未满足项": {
      "数值": [
        "特征 '数值' 的约束 min_n_bins=3, max_n_bins=3, min_bin_size=0.01, max_bin_size=None, monotonic=descending 无法满足: 候选边界上不存在同时满足箱数、样本数、坏率及单调约束的可行分箱"
      ]
    }
  }
]


## 4. 72 组真实数据对照

三个特征 × 三种支持目标的方法 × 八个复合指标。每组分别按名称和等价函数拟合（合计 144 次），检查规则一致，以及逐箱人数、坏数与原始行聚合一致。遗传分箱固定随机种子。


In [5]:
matrix=[]
for feature in FEATURES:
    for method in ['or_tools','cp_sat','genetic']:
        controls={'population_size':12,'generations':6} if method=='genetic' else {'n_prebins':8,'time_limit':5}
        for name in CustomObjectives.names():
            params={'iv_weight':1.0}
            common=dict(method=method,max_n_bins=3,min_bin_size=.05,monotonic=False,n_jobs=1,random_state=42,**controls)
            by_name=OptimalBinning(objective=name,objective_params=params,**common).fit(DATA[[feature]],DATA.FPD)
            by_function=OptimalBinning(objective=CustomObjectives.get(name,**params),**common).fit(DATA[[feature]],DATA.FPD)
            actual=by_name.transform(DATA[[feature]])[feature].to_numpy()
            expected=by_function.transform(DATA[[feature]])[feature].to_numpy()
            np.testing.assert_array_equal(actual,expected)
            table=compute_bin_stats(actual,DATA.FPD.to_numpy(),round_digits=False)
            raw=pd.DataFrame({'分箱':actual,'标签':DATA.FPD}).groupby('分箱')['标签'].agg(['size','sum'])
            got=table.set_index('分箱').loc[raw.index]
            np.testing.assert_array_equal(got['样本总数'],raw['size'])
            np.testing.assert_array_equal(got['坏样本数'],raw['sum'])
            assert int(table['样本总数'].sum())==len(DATA)
            matrix.append({'字段':feature,'方法':method,'评价目标':name,'普通箱数':len(raw),'样本数':int(raw['size'].sum()),
                           'IV':float(table['分档IV值'].sum()),'名称与函数规则一致':True})
    print('完成字段：',feature)
EVIDENCE['真实数据名称与函数对照']=matrix
save()
assert len(matrix)==72
print(pd.DataFrame(matrix).head(12).to_string(index=False))


完成字段： 衡枢鉴真分老客版


完成字段： 近六个月非银多头机构数


完成字段： 青云24
      字段       方法                 评价目标  普通箱数  样本数     IV  名称与函数规则一致
衡枢鉴真分老客版 or_tools          max_lift_iv     3  970 0.2528       True
衡枢鉴真分老客版 or_tools          min_lift_iv     3  970 0.2528       True
衡枢鉴真分老客版 or_tools  max_min_lift_sum_iv     3  970 0.2528       True
衡枢鉴真分老客版 or_tools max_lift_distance_iv     3  970 0.2528       True
衡枢鉴真分老客版 or_tools min_lift_distance_iv     3  970 0.2528       True
衡枢鉴真分老客版 or_tools lift_distance_sum_iv     3  970 0.2528       True
衡枢鉴真分老客版 or_tools            max_ks_iv     3  970 0.2528       True
衡枢鉴真分老客版 or_tools      woe_variance_iv     3  970 0.2528       True
衡枢鉴真分老客版   cp_sat          max_lift_iv     3  970 0.2528       True
衡枢鉴真分老客版   cp_sat          min_lift_iv     3  970 0.2528       True
衡枢鉴真分老客版   cp_sat  max_min_lift_sum_iv     3  970 0.2528       True
衡枢鉴真分老客版   cp_sat max_lift_distance_iv     3  970 0.2528       True


## 5. 预定义 IV 的独立核算

两箱好坏数分别为 `(4,1)`、`(1,4)`，总体好坏各 5 人。标准 IV 为 `2 × (0.8−0.2) × ln(0.8/0.2)`，应为正值。复合目标关闭附加项后必须得到同一 IV。


In [6]:
checks=[]
for case,bins,y,manual in [
    ('等人数两箱',np.repeat([0,1],5),np.asarray([0,0,0,0,1,0,1,1,1,1]),1.2*np.log(4)),
    ('不等人数两箱',np.r_[np.zeros(5),np.ones(15)].astype(int),np.r_[np.zeros(4),[1],np.zeros(6),np.ones(9)].astype(int),.3*np.log(6))]:
    table=compute_bin_stats(bins,y,round_digits=False)
    rows=[]
    for r in table.to_dict('records'):
        rows.append({**r,'count':int(r['样本总数']),'good':int(r['好样本数']),'bad':int(r['坏样本数']),
                     'iv':r['分档IV值'],'woe':r['分档WOE值'],'lift':r['LIFT值']})
    for name in CustomObjectives.names():
        key='ks_weight' if name=='max_ks_iv' else 'variance_weight' if name=='woe_variance_iv' else 'lift_weight'
        value=CustomObjectives.get(name,**{key:0.,'iv_weight':1.})(rows,int((y==0).sum()),int((y==1).sum()))
        np.testing.assert_allclose(value,manual)
        checks.append({'样本结构':case,'目标':name,'复合目标的IV项':float(value),'独立公式IV':float(manual)})
EVIDENCE['手算IV核验']=checks
print(pd.DataFrame(checks).to_string(index=False))


  样本结构                   目标  复合目标的IV项  独立公式IV
 等人数两箱          max_lift_iv    1.6636  1.6636
 等人数两箱          min_lift_iv    1.6636  1.6636
 等人数两箱  max_min_lift_sum_iv    1.6636  1.6636
 等人数两箱 max_lift_distance_iv    1.6636  1.6636
 等人数两箱 min_lift_distance_iv    1.6636  1.6636
 等人数两箱 lift_distance_sum_iv    1.6636  1.6636
 等人数两箱            max_ks_iv    1.6636  1.6636
 等人数两箱      woe_variance_iv    1.6636  1.6636
不等人数两箱          max_lift_iv    0.5375  0.5375
不等人数两箱          min_lift_iv    0.5375  0.5375
不等人数两箱  max_min_lift_sum_iv    0.5375  0.5375
不等人数两箱 max_lift_distance_iv    0.5375  0.5375
不等人数两箱 min_lift_distance_iv    0.5375  0.5375
不等人数两箱 lift_distance_sum_iv    0.5375  0.5375
不等人数两箱            max_ks_iv    0.5375  0.5375
不等人数两箱      woe_variance_iv    0.5375  0.5375


## 6. hsbin 结构化调用

名称和参数字典可以直接用于已有 `optimal_binning_fit` 操作；Python 回调用于本地 Python/Notebook。此处使用当前解释器，不安装缺失依赖。产物写入本项目临时验证目录。


In [7]:
from hscredit.skills_runtime import execute_skill
request={'version':'1','operation':'optimal_binning_fit',
    'inputs':{'data':{'kind':'object_ref','ref':'loans'}},
    'parameters':{'features':[FEATURES[0]],'target':'FPD','method':'or_tools','max_n_bins':3,'min_bin_size':.05,
                  'objective':'max_lift_iv','objective_params':{'lift_weight':.2},'constraint_policy':'legacy','n_jobs':1},
    'output':{'directory':str(ROOT/'.audit_tmp/binning-objective-policy-artifacts'),'name':'分箱目标示例','overwrite':True},
    'environment':{'mode':'current','install_missing':False}}
result=execute_skill('hsbin',request,objects={'loans':DATA})
assert result['status']=='success',result
EVIDENCE['hsbin结构化调用']={'状态':result['status'],'产物数量':len(result['artifacts'])}
print('调用完成，产物数量：',len(result['artifacts']))


调用完成，产物数量： 2


## 7. 类别与二维轴的真实数据检查

类别变量沿用训练期类别顺序和未知类别协议；二维通过轴参数选择已支持的评价目标，区域合并仍采用原有准则。

In [8]:
category_checks=[]
for method in ['or_tools','cp_sat','genetic']:
    controls={'population_size':12,'generations':6} if method=='genetic' else {'n_prebins':8,'time_limit':5}
    model=OptimalBinning(method=method,objective='max_ks_iv',objective_params={'ks_weight':.2},
        max_n_bins=3,min_bin_size=.05,n_jobs=1,random_state=42,**controls).fit(DATA[['商品类别']],DATA.FPD)
    ids=model.transform(DATA[['商品类别']])['商品类别']
    table=compute_bin_stats(ids,DATA.FPD,round_digits=False)
    assert int(table['样本总数'].sum())==len(DATA)
    assert int(table['坏样本数'].sum())==int(DATA.FPD.sum())
    unknown=model.transform(pd.DataFrame({'商品类别':['核验专用未知类别']})).iloc[0,0]
    assert unknown==-3
    category_checks.append({'方法':method,'样本数':int(table['样本总数'].sum()),'坏样本数':int(table['坏样本数'].sum()),'实际箱数':model.n_bins_['商品类别']})
model2=OptimalBinning2D(max_n_bins=3,min_bin_size=.05,n_jobs=1,
    x_params={'method':'cp_sat','objective':'max_lift_iv','objective_params':{'lift_weight':.2},'n_prebins':8,'time_limit':5}).fit(DATA[FEATURES[:2]],DATA.FPD)
ids2=np.asarray(model2.transform(DATA[FEATURES[:2]],metric='indices')).reshape(-1)
table2=compute_bin_stats(ids2,DATA.FPD,round_digits=False)
assert int(table2['样本总数'].sum())==len(DATA)
EVIDENCE['类别与二维核验']={'类别':category_checks,'二维':{'样本数':int(table2['样本总数'].sum()),'实际箱数':model2.n_bins_2d_,'轴目标':model2.binner_x_.objective}}
print(json.dumps(EVIDENCE['类别与二维核验'],ensure_ascii=False,indent=2))


{
  "类别": [
    {
      "方法": "or_tools",
      "样本数": 970,
      "坏样本数": 136,
      "实际箱数": 3
    },
    {
      "方法": "cp_sat",
      "样本数": 970,
      "坏样本数": 136,
      "实际箱数": 3
    },
    {
      "方法": "genetic",
      "样本数": 970,
      "坏样本数": 136,
      "实际箱数": 3
    }
  ],
  "二维": {
    "样本数": 970,
    "实际箱数": 3,
    "轴目标": "max_lift_iv"
  }
}


## 8. 记录验证范围和源码版本

不将重叠测试数量相加；该记录用于复现本次 API 扩展。业务分箱表仍使用原有中文指标列。

In [9]:
xml_path=ROOT/'.audit_tmp/binning-objective-policy-final.xml'
if xml_path.exists():
    suite=ET.parse(xml_path).getroot().find('testsuite')
    EVIDENCE['测试记录']={'日期':'2026-10-27','解释器':sys.executable,
        '范围':['tests/test_binning/','tests/test_parallel_api_contract.py','tests/test_skills/test_hsbin_binners.py','tests/test_skills/test_hsbin_tables.py','tests/test_docs_parameter_rendering.py','tests/test_docs_build_validation.py'],
        **{k:int(suite.get(k,0)) for k in ['tests','failures','errors','skipped']},'耗时秒':float(suite.get('time'))}
    assert EVIDENCE['测试记录']['failures']==EVIDENCE['测试记录']['errors']==0
for name,digest in EVIDENCE['源码SHA256'].items():
    assert hashlib.sha256((ROOT/name).read_bytes()).hexdigest()==digest
EVIDENCE['所有核验完成']=True
save()
print('核验完成，结果文件：',OUT/'binning-objectives-and-constraints-2026-10-27.json')


核验完成，结果文件： E:\workspace\hscredit\docs\validation\binning-objectives-and-constraints-2026-10-27.json
